# Production Pipeline — KNN Anonymization

Standalone notebook (no external `.py` files).

- **Batch mode:** reads `parameter_combinations.csv`, runs every row, writes `results/experiment_ranking.csv`
- **Single mode:** set `RUN_MODE = "single"`, edit USER CONFIG, exports to `output/`
- **Targets:** set `TARGET_COLS` to one or many columns (categorical or numeric; auto-detected)

Flow: load data → k-anonymity → KNN synthesis → validation → ranking export


In [1]:
# --- imports & paths ---
import json
import time
import tracemalloc
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency, ks_2samp
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import f1_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, RobustScaler, StandardScaler

PIPELINE_ROOT = Path(".").resolve()
DATA_PATH = PIPELINE_ROOT.parent / "Bank Customer Churn Prediction.csv"
RESULTS_DIR = PIPELINE_ROOT / "results"
OUTPUT_DIR = PIPELINE_ROOT / "output"

ID_COLS = ["customer_id"]
CATEGORICAL_COLS = ["country", "gender"]
NUMERICAL_COLS = [
    "credit_score", "age", "tenure", "balance",
    "products_number", "credit_card", "active_member", "estimated_salary",
]
QI_COLS = CATEGORICAL_COLS + NUMERICAL_COLS
KANON_QI_COLS = CATEGORICAL_COLS + ["age", "tenure", "products_number", "credit_card", "active_member"]

# One or many targets — examples:
# TARGET_COLS = ["churn"]
# TARGET_COLS = ["churn", "risk_score"]
TARGET_COLS = ["churn"]
TARGET_KIND = {}  # optional override per column: {"churn": "categorical", "risk_score": "numeric"}
CAT_TARGET_MAX_UNIQUE = 20  # numeric columns with <= this many unique values → categorical

MISSING_LABEL = "Missing"
TVD_THRESHOLD = 0.10
KS_THRESHOLD = 0.10
PASS_RATE_TARGET = 0.85
UTILITY_THRESHOLD = 0.80  # synthetic utility must be >= this fraction of baseline

sns.set_style("whitegrid")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready — data: {DATA_PATH.name}")


Ready — data: Bank Customer Churn Prediction.csv


## CONFIG


In [2]:
# batch | single
RUN_MODE = "batch"

PARAMETER_COMBINATIONS_CSV = PIPELINE_ROOT.parent / "final_folder" / "parameter_combinations_filtered.csv"
EXPERIMENT_RANKING_CSV = RESULTS_DIR / "experiment_ranking.csv"

RANDOM_STATE = 42
CHECKPOINT_EVERY = 25
BATCH_START = 0
BATCH_LIMIT = None       # e.g. 10 for quick test
EXPORT_TOP_TO_OUTPUT = True

# Single-run manual settings
K_ANONYMITY = 5
K_NEIGHBORS = 15
NUM_WEIGHT = 1.0
CAT_WEIGHT = 1.0
SCALER_METHOD = "robust"
CAT_GEN_METHOD = "weighted_mode"
NUM_GEN_METHOD = "interpolation"
TARGET_GEN_METHOD = "probability"
DISTANCE_MODE = "weighted_sum"
NUM_DISTANCE_METRIC = "euclidean"
CAT_DISTANCE_METRIC = "hamming"
MINKOWSKI_P = 3
RUN_NAME = "Production Run"

pd.DataFrame({
    "setting": ["RUN_MODE", "TARGET_COLS", "PARAMETER_COMBINATIONS_CSV", "BATCH_LIMIT"],
    "value": [RUN_MODE, str(TARGET_COLS), str(PARAMETER_COMBINATIONS_CSV), BATCH_LIMIT],
})


,setting,value
0,RUN_MODE,batch
1,TARGET_COLS,['churn']
2,PARAMETER_COMBINATIONS_CSV,/home/neosoft/KNN_demo/final_folder/parameter_...
3,BATCH_LIMIT,NaN


## Pipeline functions (all logic lives here)


In [3]:
def normalize_target_cols(target_cols):
    if isinstance(target_cols, str):
        return [target_cols]
    return list(target_cols)


def classify_targets(df, target_cols, kind_overrides=None):
    kind_overrides = kind_overrides or {}
    kinds = {}
    for col in target_cols:
        if col not in df.columns:
            raise ValueError(f"Target column not in dataset: {col}")
        if col in kind_overrides:
            kinds[col] = kind_overrides[col]
        elif df[col].dtype == object or str(df[col].dtype) == "category":
            kinds[col] = "categorical"
        elif pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() <= CAT_TARGET_MAX_UNIQUE:
            kinds[col] = "categorical"
        else:
            kinds[col] = "numeric"
    cat_targets = [c for c in target_cols if kinds[c] == "categorical"]
    num_targets = [c for c in target_cols if kinds[c] == "numeric"]
    return kinds, cat_targets, num_targets


def target_bounds(df, cols):
  return {
      c: {
          "p01": float(np.percentile(df[c].dropna(), 1)),
          "p99": float(np.percentile(df[c].dropna(), 99)),
          "median": float(df[c].median()),
      }
      for c in cols
  }


def generalize_for_kanonymity(df, qi_cols):
    g = df[qi_cols].copy()
    if "credit_score" in g.columns:
        g["credit_score"] = (g["credit_score"] // 50) * 50
    if "age" in g.columns:
        g["age"] = (g["age"] // 5) * 5
    if "balance" in g.columns:
        g["balance"] = pd.qcut(g["balance"].rank(method="first"), q=20, duplicates="drop").astype(str)
    if "estimated_salary" in g.columns:
        g["estimated_salary"] = pd.qcut(g["estimated_salary"].rank(method="first"), q=20, duplicates="drop").astype(str)
    for col in g.select_dtypes(include="object").columns:
        g[col] = g[col].astype(str).fillna(MISSING_LABEL)
    return g


def identify_suppressed(df, k_anonymity):
    generalized = generalize_for_kanonymity(df, KANON_QI_COLS)
    class_sizes = generalized.groupby(list(generalized.columns), dropna=False).transform("size")
    suppressed = class_sizes < k_anonymity
    pool_idx = np.where(~suppressed.values)[0]
    synth_idx = np.where(suppressed.values)[0]
    if len(pool_idx) == 0:
        raise ValueError(f"k_anonymity={k_anonymity}: no neighbour pool")
    return suppressed, pool_idx, synth_idx


def fit_preprocessors(df, scaler_method):
    cat_encoders = {}
    for col in CATEGORICAL_COLS:
        enc = LabelEncoder()
        enc.fit(df[col].astype(str).fillna(MISSING_LABEL))
        cat_encoders[col] = enc
    num_medians = df[NUMERICAL_COLS].median()
    num_filled = df[NUMERICAL_COLS].fillna(num_medians).values.astype(float)
    scaler_cls = {"standard": StandardScaler, "minmax": MinMaxScaler, "robust": RobustScaler}[scaler_method]
    num_scaler = scaler_cls().fit(num_filled)
    num_p01 = {c: np.percentile(df[c].dropna(), 1) for c in NUMERICAL_COLS}
    num_p99 = {c: np.percentile(df[c].dropna(), 99) for c in NUMERICAL_COLS}
    X_cat = np.column_stack([
        cat_encoders[c].transform(df[c].astype(str).fillna(MISSING_LABEL)) for c in CATEGORICAL_COLS
    ])
    X_num = num_scaler.transform(num_filled)
    num_ranges = np.ptp(num_filled, axis=0)
    return {
        "cat_encoders": cat_encoders,
        "num_medians": num_medians,
        "num_scaler": num_scaler,
        "num_p01": num_p01,
        "num_p99": num_p99,
        "X_cat": X_cat,
        "X_num": X_num,
        "num_filled": num_filled,
        "num_ranges": num_ranges,
    }


def categorical_pairwise_distance(pool_cat, synth_cat, metric):
    mismatch = pool_cat[np.newaxis, :, :] != synth_cat[:, np.newaxis, :]
    n_cat = pool_cat.shape[1]
    if n_cat == 0:
        return np.zeros((len(synth_cat), len(pool_cat)))
    matches = (~mismatch).sum(axis=2).astype(float)
    if metric == "hamming":
        return np.mean(mismatch, axis=2)
    if metric == "jaccard":
        union = 2.0 * n_cat - matches
        with np.errstate(divide="ignore", invalid="ignore"):
            sim = np.where(union > 0, matches / union, 1.0)
        return 1.0 - sim
    if metric == "overlap":
        return 1.0 - matches / n_cat
    raise ValueError(f"Unknown cat distance: {metric}")


def build_neighbor_cache(cfg, prep, pool_idx, synth_idx):
    pool_cat = prep["X_cat"][pool_idx]
    synth_cat = prep["X_cat"][synth_idx]
    if cfg["distance_mode"] == "gower":
        pool_num = prep["num_filled"][pool_idx]
        synth_num = prep["num_filled"][synth_idx]
        n_synth, n_pool = len(synth_num), len(pool_num)
        total_dist = np.zeros((n_synth, n_pool))
        safe_ranges = np.where(prep["num_ranges"] < 1e-8, 1.0, prep["num_ranges"])
        for j in range(pool_num.shape[1]):
            total_dist += np.abs(pool_num[np.newaxis, :, j] - synth_num[:, np.newaxis, j]) / safe_ranges[j]
        for j in range(pool_cat.shape[1]):
            total_dist += (pool_cat[np.newaxis, :, j] != synth_cat[:, np.newaxis, j]).astype(float)
        total_dist /= (pool_num.shape[1] + pool_cat.shape[1])
    else:
        pool_num = prep["X_num"][pool_idx]
        synth_num = prep["X_num"][synth_idx]
        diff = pool_num[np.newaxis, :, :] - synth_num[:, np.newaxis, :]
        metric = cfg["num_distance_metric"]
        if metric == "euclidean":
            num_dist = np.linalg.norm(diff, axis=2)
        elif metric == "manhattan":
            num_dist = np.sum(np.abs(diff), axis=2)
        elif metric == "minkowski":
            p = int(cfg["minkowski_p"])
            num_dist = np.sum(np.abs(diff) ** p, axis=2) ** (1.0 / p)
        else:
            raise ValueError(metric)
        cat_dist = categorical_pairwise_distance(pool_cat, synth_cat, cfg["cat_distance_metric"])
        total_dist = float(cfg["num_weight"]) * num_dist + float(cfg["cat_weight"]) * cat_dist

    k_cache = min(int(cfg["k_neighbors"]), len(pool_idx))
    cache = {}
    for local_i, global_i in enumerate(synth_idx):
        row_dist = total_dist[local_i]
        idx_local = np.argpartition(row_dist, k_cache - 1)[:k_cache]
        order = idx_local[np.argsort(row_dist[idx_local])]
        cache[int(global_i)] = (row_dist[order], pool_idx[order])
    return cache


def generate_categorical(vals, weights, method, rng):
    if method == "mode":
        return Counter(vals).most_common(1)[0][0]
    if method == "weighted_mode":
        counts = defaultdict(float)
        for v, w in zip(vals, weights):
            counts[str(v)] += w
        return max(counts, key=counts.get)
    if method == "probability":
        return str(rng.choice(vals, p=weights / weights.sum()))
    raise ValueError(method)


def synthesize_numeric_value(row_idx, nbrs, w, col, df, X_num, col_to_j, cfg, rng, bounds):
    j = col_to_j.get(col)
    if cfg["num_gen_method"] == "interpolation" and j is not None:
        nj = rng.choice(nbrs)
        t = rng.random()
        val = X_num[row_idx, j] + t * (X_num[nj, j] - X_num[row_idx, j])
    elif cfg["num_gen_method"] == "interpolation":
        nj = rng.choice(nbrs)
        t = rng.random()
        base = float(df.loc[row_idx, col])
        nval = float(df.loc[nj, col])
        val = base + t * (nval - base)
    elif j is not None:
        val = float(np.dot(w, X_num[nbrs, j]))
    else:
        val = float(np.dot(w, df.loc[nbrs, col].astype(float).values))
    return float(np.clip(val, bounds[col]["p01"], bounds[col]["p99"]))


def synthesize_dataset(df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache,
                       target_cols, cat_target_cols, num_target_cols, target_bounds_map):
    rng = np.random.default_rng(int(cfg.get("random_state", RANDOM_STATE)))
    df_out = df.copy()
    int_cols = {c for c in NUMERICAL_COLS + num_target_cols if c in df.columns and pd.api.types.is_integer_dtype(df[c])}
    X_num = prep["X_num"]
    col_to_j = {col: j for j, col in enumerate(NUMERICAL_COLS)}

    for row_idx in synth_idx:
        row_idx = int(row_idx)
        dists_full, nbrs_full = neighbor_cache[row_idx]
        k = min(int(cfg["k_neighbors"]), len(dists_full))
        dists, nbrs = dists_full[:k], nbrs_full[:k]
        w = 1.0 / (dists + 1e-8)
        w = w / w.sum()
        row_out = {}

        for col in CATEGORICAL_COLS:
            vals = df.loc[nbrs, col].astype(str).fillna(MISSING_LABEL).values
            row_out[col] = generate_categorical(vals, w, cfg["cat_gen_method"], rng)

        synth_scaled = np.zeros(len(NUMERICAL_COLS))
        for j in range(len(NUMERICAL_COLS)):
            if cfg["num_gen_method"] == "interpolation":
                nj = rng.choice(nbrs)
                t = rng.random()
                synth_scaled[j] = X_num[row_idx, j] + t * (X_num[nj, j] - X_num[row_idx, j])
            elif cfg["num_gen_method"] == "weighted_mean":
                synth_scaled[j] = float(np.dot(w, X_num[nbrs, j]))
            else:
                raise ValueError(cfg["num_gen_method"])
        synth_num = prep["num_scaler"].inverse_transform(synth_scaled.reshape(1, -1)).flatten()
        for j, col in enumerate(NUMERICAL_COLS):
            row_out[col] = float(np.clip(synth_num[j], prep["num_p01"][col], prep["num_p99"][col]))

        for tcol in cat_target_cols:
            tvals = df.loc[nbrs, tcol].astype(str).values
            row_out[tcol] = generate_categorical(tvals, w, cfg["target_gen_method"], rng)

        for tcol in num_target_cols:
            row_out[tcol] = synthesize_numeric_value(
                row_idx, nbrs, w, tcol, df, X_num, col_to_j, cfg, rng, target_bounds_map
            )

        for col in QI_COLS + target_cols:
            val = row_out[col]
            if col in cat_target_cols:
                val = int(float(val)) if str(val).replace(".", "", 1).isdigit() else val
            elif col in int_cols:
                val = int(round(val))
            df_out.loc[row_idx, col] = val
    return df_out


def cramers_v(x, y):
    tbl = pd.crosstab(x.astype(str), y.astype(str))
    chi2 = chi2_contingency(tbl)[0]
    n = tbl.sum().sum()
    r, k = tbl.shape
    return float(np.sqrt(chi2 / (n * (min(r - 1, k - 1) + 1e-8))))


def encode_features(frame, prep, target_cols):
    out = frame[QI_COLS].copy()
    out[NUMERICAL_COLS] = out[NUMERICAL_COLS].fillna(prep["num_medians"])
    for c in CATEGORICAL_COLS:
        out[c] = prep["cat_encoders"][c].transform(out[c].astype(str).fillna(MISSING_LABEL))
    for c in target_cols:
        if c not in out.columns:
            out[c] = frame[c]
    return out


def utility_for_target(df_actual, df_out, prep, target_cols, target_kinds, train_idx, test_idx, random_state):
    baselines = {}
    synthetics = {}
    passes = {}
    for tcol in target_cols:
        kind = target_kinds[tcol]
        X_train = encode_features(df_actual.loc[train_idx], prep, target_cols)
        X_test = encode_features(df_actual.loc[test_idx], prep, target_cols)
        X_syn_train = encode_features(df_out.loc[train_idx], prep, target_cols)
        X_syn_test = encode_features(df_out.loc[test_idx], prep, target_cols)
        y_train = df_actual.loc[train_idx, tcol]
        y_test = df_actual.loc[test_idx, tcol]
        if kind == "categorical":
            model = LogisticRegression(max_iter=500, random_state=random_state)
            model.fit(X_syn_train, df_out.loc[train_idx, tcol])
            base = LogisticRegression(max_iter=500, random_state=random_state)
            base.fit(X_train, y_train)
            score_base = float(f1_score(y_test, base.predict(X_test), average="binary" if df_actual[tcol].nunique() == 2 else "weighted"))
            score_syn = float(f1_score(y_test, model.predict(X_syn_test), average="binary" if df_actual[tcol].nunique() == 2 else "weighted"))
        else:
            model = LinearRegression()
            model.fit(X_syn_train, df_out.loc[train_idx, tcol].astype(float))
            base = LinearRegression()
            base.fit(X_train, y_train.astype(float))
            score_base = float(r2_score(y_test.astype(float), base.predict(X_test)))
            score_syn = float(r2_score(y_test.astype(float), model.predict(X_syn_test)))
        baselines[tcol] = round(score_base, 4)
        synthetics[tcol] = round(score_syn, 4)
        passes[tcol] = score_syn >= UTILITY_THRESHOLD * score_base if score_base > 1e-6 else score_syn >= score_base
    return baselines, synthetics, passes


def compute_metrics(df_actual, df_out, suppressed, prep, utility_baselines, train_idx, test_idx,
                    random_state, target_cols, cat_target_cols, num_target_cols, target_kinds):
    cat_rows = []
    for col in CATEGORICAL_COLS:
        act = df_actual[col].astype(str)
        syn = df_out[col].astype(str)
        a = act.value_counts(normalize=True)
        s = syn.value_counts(normalize=True)
        tv = 0.5 * sum(abs(s.get(k, 0) - a.get(k, 0)) for k in set(a.index) | set(s.index))
        cat_rows.append({"column": col, "TVD": round(tv, 4), "pass_tvd": tv < TVD_THRESHOLD})
    for col in cat_target_cols:
        act = df_actual[col].astype(str)
        syn = df_out[col].astype(str)
        a = act.value_counts(normalize=True)
        s = syn.value_counts(normalize=True)
        tv = 0.5 * sum(abs(s.get(k, 0) - a.get(k, 0)) for k in set(a.index) | set(s.index))
        cat_rows.append({"column": col, "TVD": round(tv, 4), "pass_tvd": tv < TVD_THRESHOLD, "role": "target"})
    category_metrics = pd.DataFrame(cat_rows)
    tvd_pass_rate = float(category_metrics["pass_tvd"].mean())

    num_cols_for_ks = list(NUMERICAL_COLS) + list(num_target_cols)
    num_rows = []
    for col in num_cols_for_ks:
        act, syn = df_actual[col].astype(float), df_out[col].astype(float)
        ks_stat, ks_p = ks_2samp(act, syn)
        num_rows.append({
            "column": col,
            "KS_statistic": round(ks_stat, 4),
            "KS_pvalue": round(ks_p, 4),
            "pass_ks": ks_stat < KS_THRESHOLD,
            "role": "target" if col in num_target_cols else "qi",
        })
    numeric_metrics = pd.DataFrame(num_rows)
    ks_pass_rate = float(numeric_metrics["pass_ks"].mean())

    relationship_rows = []
    for tcol in target_cols:
        for col in CATEGORICAL_COLS:
            if target_kinds[tcol] == "categorical":
                v_a = cramers_v(df_actual[col], df_actual[tcol])
                v_s = cramers_v(df_out[col], df_out[tcol])
                relationship_rows.append({
                    "target": tcol, "column": col, "metric": "cramers_v",
                    "actual": round(v_a, 4), "synthetic": round(v_s, 4), "drift": round(abs(v_a - v_s), 4),
                })
            else:
                if pd.api.types.is_numeric_dtype(df_actual[col]):
                    c_a = df_actual[col].astype(float).corr(df_actual[tcol].astype(float))
                    c_s = df_out[col].astype(float).corr(df_out[tcol].astype(float))
                    if pd.notna(c_a) and pd.notna(c_s):
                        relationship_rows.append({
                            "target": tcol, "column": col, "metric": "correlation",
                            "actual": round(float(c_a), 4), "synthetic": round(float(c_s), 4),
                            "drift": round(abs(float(c_a) - float(c_s)), 4),
                        })
    relationship_metrics = pd.DataFrame(relationship_rows)
    mean_corr_drift = float(relationship_metrics["drift"].mean()) if len(relationship_metrics) else 0.0

    baselines, synthetics, utility_passes = utility_for_target(
        df_actual, df_out, prep, target_cols, target_kinds, train_idx, test_idx, random_state
    )
    utility_pass_rate = float(np.mean(list(utility_passes.values()))) if utility_passes else 0.0
    f1_baseline = float(np.mean(list(baselines.values()))) if baselines else 0.0
    f1_synthetic = float(np.mean(list(synthetics.values()))) if synthetics else 0.0
    utility_pass = all(utility_passes.values()) if utility_passes else False

    feature_cols = [c for c in df_actual.columns if c not in ID_COLS]
    replaced_idx = suppressed[suppressed].index
    exact_match_rate = len(
        df_out.loc[replaced_idx, feature_cols].merge(df_actual[feature_cols], how="inner")
    ) / max(len(replaced_idx), 1)

    scorecard = pd.DataFrame([
        {"area": "Quality-Categorical", "metric": "tvd_pass_rate>=0.85", "value": round(tvd_pass_rate, 4), "pass": tvd_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "ks_pass_rate>=0.85", "value": round(ks_pass_rate, 4), "pass": ks_pass_rate >= PASS_RATE_TARGET},
        {"area": "Quality-Numerical", "metric": "mean_KS<0.10", "value": round(numeric_metrics["KS_statistic"].mean(), 4), "pass": numeric_metrics["KS_statistic"].mean() < KS_THRESHOLD},
        {"area": "Relationships", "metric": "mean_corr_drift<0.05", "value": round(mean_corr_drift, 4), "pass": mean_corr_drift < 0.05},
        {"area": "Utility", "metric": f"utility>={UTILITY_THRESHOLD:.0%} baseline (all targets)", "value": round(f1_synthetic, 4), "pass": utility_pass},
        {"area": "Privacy", "metric": "replaced_exact_match_rate<0.001", "value": round(exact_match_rate, 6), "pass": exact_match_rate < 0.001},
        {"area": "Suppression", "metric": "recovery_rate", "value": 1.0, "pass": suppressed.sum() > 0},
    ])
    overall_pass = bool(scorecard["pass"].all())

    return {
        "category_metrics": category_metrics,
        "numeric_metrics": numeric_metrics,
        "relationship_metrics": relationship_metrics,
        "scorecard": scorecard,
        "overall_pass": overall_pass,
        "tvd_pass_rate": round(tvd_pass_rate, 4),
        "ks_pass_rate": round(ks_pass_rate, 4),
        "mean_tvd": round(category_metrics["TVD"].mean(), 4),
        "mean_ks": round(numeric_metrics["KS_statistic"].mean(), 4),
        "mean_corr_drift": round(mean_corr_drift, 6),
        "f1_baseline": round(f1_baseline, 4),
        "f1_synthetic": round(f1_synthetic, 4),
        "utility_baselines": baselines,
        "utility_synthetics": synthetics,
        "utility_pass_rate": round(utility_pass_rate, 4),
        "exact_match_rate": round(exact_match_rate, 6),
        "target_cols": target_cols,
    }


def is_valid_config(cfg):
    """Drop redundant grid rows: dead Gower neighbour params and invalid minkowski_p."""
    if cfg["distance_mode"] == "gower":
        if cfg["cat_distance_metric"] != "hamming":
            return False
        if cfg["scaler_method"] != "minmax":
            return False
        if cfg["num_distance_metric"] != "euclidean":
            return False
        if float(cfg["num_weight"]) != 1.0 or float(cfg["cat_weight"]) != 1.0:
            return False
    if cfg["num_distance_metric"] != "minkowski" and int(cfg["minkowski_p"]) != 3:
        return False
    return True


def config_to_key(cfg):
    k = int(cfg["k_anonymity"])
    mode = cfg["distance_mode"]
    if mode == "gower":
        return (k, mode)
    return (
        k, cfg["scaler_method"], mode,
        cfg["num_distance_metric"], cfg["cat_distance_metric"], int(cfg["minkowski_p"]),
        float(cfg["num_weight"]), float(cfg["cat_weight"]),
    )


def make_folder_name(idx, cfg):
    cat_tag = f"-cat-{cfg['cat_distance_metric']}" if cfg["distance_mode"] == "weighted_sum" else ""
    return (
        f"{idx:03d}_k{int(cfg['k_neighbors'])}_cat-{cfg['cat_gen_method']}_"
        f"num-{cfg['num_gen_method']}_tgt-{cfg['target_gen_method']}_"
        f"scale-{cfg['scaler_method']}_{cfg['distance_mode']}-"
        f"{cfg['num_distance_metric']}{cat_tag}_w-{cfg['distance_profile']}"
    )


def make_display_name(cfg):
    cat_m = cfg["cat_distance_metric"] if cfg["distance_mode"] == "weighted_sum" else "gower"
    return (
        f"K={int(cfg['k_neighbors'])} cat={cfg['cat_gen_method']} num={cfg['num_gen_method']} "
        f"tgt={cfg['target_gen_method']} scale={cfg['scaler_method']} "
        f"{cfg['distance_mode']}/{cfg['num_distance_metric']}/{cat_m}/{cfg['distance_profile']}"
    )


def finalize_ranking(rows):
    ranking = pd.DataFrame(rows)
    ranking["composite_score"] = (
        ranking["tvd_pass_rate"]
        + ranking["ks_pass_rate"]
        + ranking["f1_synthetic"] / ranking["f1_baseline"].clip(lower=0.01)
        - ranking["mean_ks"]
        - ranking["runtime_sec"] / 100
    )
    ranking = ranking.sort_values("composite_score", ascending=False).reset_index(drop=True)
    ranking["rank"] = ranking.index + 1
    return ranking


def row_to_config(row):
    return {
        "k_anonymity": int(row["k_anonymity"]),
        "k_neighbors": int(row["k_neighbors"]),
        "cat_gen_method": str(row["cat_gen_method"]),
        "num_gen_method": str(row["num_gen_method"]),
        "target_gen_method": str(row["target_gen_method"]),
        "scaler_method": str(row["scaler_method"]),
        "distance_mode": str(row["distance_mode"]),
        "num_distance_metric": str(row["num_distance_metric"]),
        "minkowski_p": int(row["minkowski_p"]),
        "cat_distance_metric": str(row["cat_distance_metric"]),
        "num_weight": float(row["num_weight"]),
        "cat_weight": float(row["cat_weight"]),
        "distance_profile": str(row["distance_profile"]),
        "random_state": RANDOM_STATE,
    }


def export_production_outputs(df_out, metrics, cfg, target_cols, run_name="Production Run"):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(OUTPUT_DIR / "anonymized_dataset.csv", index=False)
    metrics["category_metrics"].to_csv(OUTPUT_DIR / "category_metrics.csv", index=False)
    metrics["numeric_metrics"].to_csv(OUTPUT_DIR / "numeric_metrics.csv", index=False)
    metrics["relationship_metrics"].to_csv(OUTPUT_DIR / "relationship_metrics.csv", index=False)
    metrics["scorecard"].to_csv(OUTPUT_DIR / "scorecard.csv", index=False)
    summary = {
        "runtime_sec": metrics.get("runtime_sec"),
        "peak_memory_mb": metrics.get("peak_memory_mb"),
        "n_suppressed": metrics.get("n_suppressed"),
        "overall_pass": metrics.get("overall_pass"),
        "target_cols": target_cols,
        "utility_baselines": metrics.get("utility_baselines"),
        "utility_synthetics": metrics.get("utility_synthetics"),
    }
    with open(OUTPUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    user_config = {
        "name": run_name,
        "target_cols": target_cols,
        "k_anonymity": int(cfg["k_anonymity"]),
        "k_neighbors": int(cfg["k_neighbors"]),
        "num_weight": float(cfg["num_weight"]),
        "cat_weight": float(cfg["cat_weight"]),
        "scaler_method": cfg["scaler_method"],
        "cat_gen_method": cfg["cat_gen_method"],
        "num_gen_method": cfg["num_gen_method"],
        "target_gen_method": cfg["target_gen_method"],
        "distance_mode": cfg["distance_mode"],
        "num_distance_metric": cfg["num_distance_metric"],
        "cat_distance_metric": cfg["cat_distance_metric"],
        "minkowski_p": int(cfg["minkowski_p"]),
        "random_state": int(cfg.get("random_state", RANDOM_STATE)),
    }
    with open(OUTPUT_DIR / "config.json", "w") as f:
        json.dump(user_config, f, indent=2)
    with open(OUTPUT_DIR / "user_config.json", "w") as f:
        json.dump(user_config, f, indent=2)
    print(f"Exported production outputs → {OUTPUT_DIR}/")


print("Pipeline functions loaded.")


Pipeline functions loaded.


## 1 — Load data & resolve targets


In [4]:
df = pd.read_csv(DATA_PATH)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.drop(columns=[c for c in df.columns if df[c].isna().all()]).reset_index(drop=True)

TARGET_COLS = normalize_target_cols(TARGET_COLS)
TARGET_KINDS, CAT_TARGET_COLS, NUM_TARGET_COLS = classify_targets(df, TARGET_COLS, TARGET_KIND)
TARGET_BOUNDS = target_bounds(df, NUM_TARGET_COLS)

print(f"Loaded {len(df):,} rows")
print(f"Targets ({len(TARGET_COLS)}): {TARGET_COLS}")
print(f"  categorical: {CAT_TARGET_COLS}")
print(f"  numeric: {NUM_TARGET_COLS}")

train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=RANDOM_STATE)
prep_baseline = fit_preprocessors(df, "standard")

UTILITY_BASELINES, _, _ = utility_for_target(
    df, df, prep_baseline, TARGET_COLS, TARGET_KINDS, train_idx, test_idx, RANDOM_STATE
)
print("Utility baselines (actual data):", UTILITY_BASELINES)


Loaded 10,000 rows
Targets (1): ['churn']
  categorical: ['churn']
  numeric: []


/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Utility baselines (actual data): {'churn': 1.0}


/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 2 — Run experiments


In [ ]:
suppression_cache = {}
prep_cache = {}
neighbor_cache_store = {}
ranking_rows = []

if EXPERIMENT_RANKING_CSV.exists():
    existing = pd.read_csv(EXPERIMENT_RANKING_CSV)
    done_folders = set(existing["folder"].astype(str))
    ranking_rows = existing.to_dict("records")
    print(f"Resuming: {len(done_folders)} configs already in {EXPERIMENT_RANKING_CSV.name}")
else:
    done_folders = set()

if RUN_MODE == "single":
    combos = pd.DataFrame([{
        "k_anonymity": K_ANONYMITY,
        "k_neighbors": K_NEIGHBORS,
        "cat_gen_method": CAT_GEN_METHOD,
        "num_gen_method": NUM_GEN_METHOD,
        "target_gen_method": TARGET_GEN_METHOD,
        "scaler_method": SCALER_METHOD,
        "distance_mode": DISTANCE_MODE,
        "num_distance_metric": NUM_DISTANCE_METRIC,
        "minkowski_p": MINKOWSKI_P,
        "cat_distance_metric": CAT_DISTANCE_METRIC,
        "num_weight": NUM_WEIGHT,
        "cat_weight": CAT_WEIGHT,
        "distance_profile": "balanced",
    }])
else:
    combos = pd.read_csv(PARAMETER_COMBINATIONS_CSV)
    end = len(combos) if BATCH_LIMIT is None else min(len(combos), BATCH_START + BATCH_LIMIT)
    combos = combos.iloc[BATCH_START:end].reset_index(drop=True)
    print(f"Batch mode: {len(combos):,} configs from {PARAMETER_COMBINATIONS_CSV.name}")

for i, row in combos.iterrows():
    cfg = row_to_config(row)
    folder = make_folder_name(i + 1 + BATCH_START, cfg)
    if folder in done_folders:
        continue

    print(f"[{i+1}/{len(combos)}] {folder}")
    tracemalloc.start()
    t0 = time.perf_counter()

    k = int(cfg["k_anonymity"])
    if k not in suppression_cache:
        suppression_cache[k] = identify_suppressed(df, k)
    suppressed, pool_idx, synth_idx = suppression_cache[k]

    scaler = cfg["scaler_method"]
    if scaler not in prep_cache:
        prep_cache[scaler] = fit_preprocessors(df, scaler)
    prep = prep_cache[scaler]

    nkey = config_to_key(cfg)
    if nkey not in neighbor_cache_store:
        neighbor_cache_store[nkey] = build_neighbor_cache(cfg, prep, pool_idx, synth_idx)
    neighbor_cache = neighbor_cache_store[nkey]

    df_out = synthesize_dataset(
        df, cfg, prep, suppressed, pool_idx, synth_idx, neighbor_cache,
        TARGET_COLS, CAT_TARGET_COLS, NUM_TARGET_COLS, TARGET_BOUNDS,
    )
    metrics = compute_metrics(
        df, df_out, suppressed, prep, UTILITY_BASELINES, train_idx, test_idx, RANDOM_STATE,
        TARGET_COLS, CAT_TARGET_COLS, NUM_TARGET_COLS, TARGET_KINDS,
    )

    runtime_sec = round(time.perf_counter() - t0, 3)
    _, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    ranking_rows.append({
        "folder": folder,
        "name": make_display_name(cfg),
        "target_cols": "|".join(TARGET_COLS),
        "k_anonymity": k,
        "k_neighbors": int(cfg["k_neighbors"]),
        "cat_gen_method": cfg["cat_gen_method"],
        "num_gen_method": cfg["num_gen_method"],
        "target_gen_method": cfg["target_gen_method"],
        "scaler_method": scaler,
        "num_weight": float(cfg["num_weight"]),
        "cat_weight": float(cfg["cat_weight"]),
        "distance_profile": cfg["distance_profile"],
        "distance_mode": cfg["distance_mode"],
        "num_distance_metric": cfg["num_distance_metric"],
        "cat_distance_metric": cfg["cat_distance_metric"],
        "minkowski_p": int(cfg["minkowski_p"]),
        "random_state": int(cfg.get("random_state", RANDOM_STATE)),
        "runtime_sec": runtime_sec,
        "peak_memory_mb": round(peak_mem / 1024 / 1024, 2),
        "n_suppressed": int(suppressed.sum()),
        "tvd_pass_rate": metrics["tvd_pass_rate"],
        "ks_pass_rate": metrics["ks_pass_rate"],
        "mean_tvd": metrics["mean_tvd"],
        "mean_ks": metrics["mean_ks"],
        "mean_corr_drift": metrics["mean_corr_drift"],
        "f1_baseline": metrics["f1_baseline"],
        "f1_synthetic": metrics["f1_synthetic"],
        "utility_pass_rate": metrics["utility_pass_rate"],
        "exact_match_rate": metrics["exact_match_rate"],
        "overall_pass": metrics["overall_pass"],
    })
    done_folders.add(folder)

    if len(ranking_rows) % CHECKPOINT_EVERY == 0:
        finalize_ranking(ranking_rows).to_csv(EXPERIMENT_RANKING_CSV, index=False)
        print(f"  checkpoint → {EXPERIMENT_RANKING_CSV}")

    if RUN_MODE == "single":
        last_df_out = df_out
        last_metrics = metrics
        last_metrics["runtime_sec"] = runtime_sec
        last_metrics["peak_memory_mb"] = round(peak_mem / 1024 / 1024, 2)
        last_metrics["n_suppressed"] = int(suppressed.sum())
        last_cfg = cfg

ranking = finalize_ranking(ranking_rows)
ranking.to_csv(EXPERIMENT_RANKING_CSV, index=False)
print(f"\nSaved {len(ranking):,} rows → {EXPERIMENT_RANKING_CSV}")
if len(ranking):
    print(f"Top: {ranking.iloc[0]['folder']} | overall_pass={ranking.iloc[0]['overall_pass']}")
    print(f"Passing all checks: {int(ranking['overall_pass'].sum())}/{len(ranking)}")
ranking.head(10)


/tmp/ipykernel_654116/2790866539.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in g.select_dtypes(include="object").columns:


Resuming: 13 configs already in experiment_ranking.csv
Batch mode: 7,848 configs from parameter_combinations_filtered.csv
[2/7848] 002_k3_cat-mode_num-weighted_mean_tgt-mode_scale-standard_weighted_sum-euclidean-cat-hamming_w-balanced


/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1

[3/7848] 003_k3_cat-mode_num-weighted_mean_tgt-mode_scale-standard_weighted_sum-euclidean-cat-hamming_w-balanced


/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic

...

[8/7848] 008_k3_cat-mode_num-weighted_mean_tgt-mode_scale-standard_weighted_sum-euclidean-cat-jaccard_w-balanced


/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/pre

[9/7848] 009_k3_cat-mode_num-weighted_mean_tgt-mode_scale-standard_weighted_sum-euclidean-cat-overlap_w-balanced


/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: ks_2samp: Exact calculation unsuccessful. Switching to method=asymp.
  res = hypotest_fun_out(*samples, **kwds)
/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/neosoft/KNN_demo/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1

[10/7848] 010_k3_cat-mode_num-weighted_mean_tgt-mode_scale-standard_weighted_sum-euclidean-cat-overlap_w-balanced


## 3 — Export best config to `output/` (optional)


In [ ]:
if RUN_MODE == "single":
    export_production_outputs(last_df_out, last_metrics, last_cfg, TARGET_COLS, RUN_NAME)
elif EXPORT_TOP_TO_OUTPUT and len(ranking):
    top = ranking.iloc[0]
    top_cfg = {
        "k_anonymity": int(top["k_anonymity"]),
        "k_neighbors": int(top["k_neighbors"]),
        "cat_gen_method": top["cat_gen_method"],
        "num_gen_method": top["num_gen_method"],
        "target_gen_method": top["target_gen_method"],
        "scaler_method": top["scaler_method"],
        "distance_mode": top["distance_mode"],
        "num_distance_metric": top["num_distance_metric"],
        "minkowski_p": int(top["minkowski_p"]),
        "cat_distance_metric": top["cat_distance_metric"],
        "num_weight": float(top["num_weight"]),
        "cat_weight": float(top["cat_weight"]),
        "distance_profile": top["distance_profile"],
        "random_state": int(top["random_state"]),
    }
    k = int(top_cfg["k_anonymity"])
    suppressed, pool_idx, synth_idx = identify_suppressed(df, k)
    prep = fit_preprocessors(df, top_cfg["scaler_method"])
    ncache = build_neighbor_cache(top_cfg, prep, pool_idx, synth_idx)
    df_top = synthesize_dataset(
        df, top_cfg, prep, suppressed, pool_idx, synth_idx, ncache,
        TARGET_COLS, CAT_TARGET_COLS, NUM_TARGET_COLS, TARGET_BOUNDS,
    )
    top_metrics = compute_metrics(
        df, df_top, suppressed, prep, UTILITY_BASELINES, train_idx, test_idx, RANDOM_STATE,
        TARGET_COLS, CAT_TARGET_COLS, NUM_TARGET_COLS, TARGET_KINDS,
    )
    top_metrics["runtime_sec"] = float(top["runtime_sec"])
    top_metrics["peak_memory_mb"] = float(top["peak_memory_mb"])
    top_metrics["n_suppressed"] = int(top["n_suppressed"])
    export_production_outputs(df_top, top_metrics, top_cfg, TARGET_COLS, run_name=str(top["name"]))
    print(f"Top config exported (rank 1): {top['folder']}")
